# FIT IBRIDO CNN + MINIMI QUADRATI

---

In [1]:
import os
import time
from datetime import datetime
from itertools import product
import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from ODMR_CNN.cnn_model import CNNPredictor
from ml_odmr.FitHybrid import FitHybrid
from ml_odmr.PoissonNoise import PoissonNoise

# =========================================================================
# 1. PARAMETRI DELLA GRIGLIA E SETUP PERCORSI
# =========================================================================
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5]

mat_file_path = "../data/10k/odmr_easyspin_raw_10000_DOPPIA_DENSITÀ.mat"

# Cartelle separate per modello puro e fit ibrido nella root del workspace
dir_model_base = os.path.abspath("../benchmarks/10k/model")
dir_hybrid_base = os.path.abspath("../benchmarks/10k/hybrid")
os.makedirs(dir_model_base, exist_ok=True)
os.makedirs(dir_hybrid_base, exist_ok=True)

random_seed = 42

# Caricamento dati dal file .mat
with h5py.File(mat_file_path, "r") as f:
    spectra_data = np.array(f["spectra_data"], dtype=np.float32).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()
    labels_c = np.array(f["labels_c"], dtype=np.float64).squeeze()
    labels_s = np.array(f["labels_s"], dtype=np.float64).squeeze()

f_ghz = np.asarray(mw_freqs, dtype=np.float64)
f_mhz = f_ghz * 1000.0
N_spectra_to_run = len(spectra_data)

# Inizializziamo una sola volta i moduli (evita di ricaricare i pesi CNN ad ogni ciclo)
noise_gen = PoissonNoise(seed=random_seed)
cnn = CNNPredictor("ODMR_CNN/artifacts/cnn_50k_48M")
hybrid_fitter = FitHybrid(w0_norm=0.025)

# Dizionari in memoria (se vuoi ispezionarli direttamente nel notebook a fine run)
predictions_model = {"c": {}, "s": {}}
predictions_hybrid = {"c": {}, "s": {}}


In [2]:
# =========================================================================
# 2. FUNZIONI DI LAVORO (GENERAZIONE, INFERENZA E FIT)
# =========================================================================
def generate_noisy_batch(contrast_val: float, n_ph_val: float) -> np.ndarray:
    """Genera il batch di spettri rumorosi per una coppia (contrast, N_ph)."""
    noisy_batch = np.empty_like(spectra_data, dtype=np.float32)
    for idx in range(N_spectra_to_run):
        ideal_sig = noise_gen.ideal_signal(spectra_data[idx], contrast=contrast_val)
        noisy_batch[idx] = noise_gen.add_noise(
            ideal_sig, n_tot=n_ph_val, seed_modifier=idx
        )
    return noisy_batch


def fit_single_wrapper(idx: int, y_noisy: np.ndarray, nu1_0: float, nu2_0: float) -> dict:
    """Esegue il fit ibrido sul singolo spettro (chiamata dai worker di Parallel)."""
    res = hybrid_fitter.fit_single(
        f_original=f_ghz,
        y_original=y_noisy,
        nu1_mhz=nu1_0,
        nu2_mhz=nu2_0,
    )
    c_t = float(labels_c[idx])
    s_t = float(labels_s[idx])

    return {
        "spectrum_id": idx,
        "converged": res.converged,
        "c_true_norm": c_t,
        "s_true_norm": s_t,
        "c_fit_norm": res.c_norm,
        "s_fit_norm": res.s_norm,
        "err_c_norm": abs(res.c_norm - c_t) if res.converged else np.nan,
        "err_s_norm": abs(res.s_norm - s_t) if res.converged else np.nan,
        "c_fit_mhz": res.c_mhz,
        "s_fit_mhz": res.s_mhz,
        "chi2_red": res.chi2_red,
        "fit_time_sec": res.duration_sec,
    }


def save_benchmark_csv(filepath: str, df: pd.DataFrame, meta: dict):
    """Salva il DataFrame su disco preceduto dall'header coi metadati."""
    header = "\n".join([f"# {k}: {v}" for k, v in meta.items()]) + "\n"
    with open(filepath, mode="w", encoding="utf-8") as f:
        f.write(header)
        df.to_csv(f, index=False, float_format="%.6f")



In [3]:
# =========================================================================
# 3. ESECUZIONE SULLA GRIGLIA (CON SALVATAGGIO SEPARATO MODEL / HYBRID)
# =========================================================================
all_combinations = list(product(list_contrast, list_Nph))
total_combos = len(all_combinations)

print(f"Totale combinazioni (Contrast x N_ph): {total_combos}")
print(f"Spettri per combinazione: {N_spectra_to_run} ({len(f_ghz)} punti per spettro)\n")

t_total_start = time.time()

# Creiamo il pool multiprocesso una sola volta per tutte le combinazioni
with Parallel(n_jobs=-1, verbose=0) as parallel:

    for combo_idx, (c_val, n_ph) in enumerate(all_combinations, start=1):
        # Sottocartelle per contrasto
        sub_model_dir = os.path.join(dir_model_base, f"contrast={c_val}")
        sub_hybrid_dir = os.path.join(dir_hybrid_base, f"contrast={c_val}")
        os.makedirs(sub_model_dir, exist_ok=True)
        os.makedirs(sub_hybrid_dir, exist_ok=True)

        file_model = os.path.join(sub_model_dir, f"cnn_Nph{int(n_ph)}.csv")
        file_hybrid = os.path.join(sub_hybrid_dir, f"hybrid_Nph{int(n_ph)}.csv")

        # Salta se entrambi i file di questa combinazione esistono già
        if os.path.exists(file_model) and os.path.exists(file_hybrid):
            print(f"[{combo_idx}/{total_combos}] SALTO (già presenti): C={c_val}, N_ph={n_ph:.1e}")
            continue

        print(f"[{combo_idx}/{total_combos}] C={c_val:<4} | N_ph={n_ph:.1e} ->", end=" ", flush=True)

        # -----------------------------------------------------------------
        # A. GENERAZIONE RUMORE
        # -----------------------------------------------------------------
        noisy_sig = generate_noisy_batch(c_val, n_ph)

        # -----------------------------------------------------------------
        # B. INFERENZA MODELLO CNN PURO (IN BATCH)
        # -----------------------------------------------------------------
        t_cnn_start = time.perf_counter()
        pred = cnn.predict_counts(noisy_sig, f_mhz)
        t_cnn_wall = time.perf_counter() - t_cnn_start
        t_cnn_per_spec = t_cnn_wall / N_spectra_to_run

        c_cnn = np.asarray(pred["c"], dtype=np.float64)
        s_cnn = np.asarray(pred["s"], dtype=np.float64)
        nu1_mhz = np.asarray(pred["nu1_MHz"], dtype=np.float64)
        nu2_mhz = np.asarray(pred["nu2_MHz"], dtype=np.float64)

        # Salvataggio nei dizionari in RAM
        predictions_model["c"][(c_val, n_ph)] = c_cnn
        predictions_model["s"][(c_val, n_ph)] = s_cnn

        # Costruzione DataFrame Modello (stesse colonne per compatibilità)
        df_model = pd.DataFrame({
            "spectrum_id": np.arange(N_spectra_to_run),
            "converged": True,  # La CNN produce sempre un output deterministico
            "c_true_norm": labels_c[:N_spectra_to_run],
            "s_true_norm": labels_s[:N_spectra_to_run],
            "c_fit_norm": c_cnn,
            "s_fit_norm": s_cnn,
            "err_c_norm": np.abs(c_cnn - labels_c[:N_spectra_to_run]),
            "err_s_norm": np.abs(s_cnn - labels_s[:N_spectra_to_run]),
            "c_fit_mhz": np.asarray(pred["center_MHz"], dtype=np.float64),
            "s_fit_mhz": np.asarray(pred["delta_MHz"], dtype=np.float64),
            "chi2_red": np.nan,
            "fit_time_sec": t_cnn_per_spec,
        })

        save_benchmark_csv(
            file_model,
            df_model,
            meta={
                "Benchmark": "1D-CNN Pure Model",
                "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Source_File": os.path.basename(mat_file_path),
                "N_spectra": N_spectra_to_run,
                "N_photons": f"{n_ph:.2e}",
                "Contrast": c_val,
                "Mean_MAE_c_norm": f"{df_model['err_c_norm'].mean():.6f}",
                "Mean_MAE_s_norm": f"{df_model['err_s_norm'].mean():.6f}",
                "Total_Batch_Time_s": f"{t_cnn_wall:.4f}",
            },
        )
        print(f"CNN: {t_cnn_wall:.2f}s |", end=" ", flush=True)

        # -----------------------------------------------------------------
        # C. FIT IBRIDO (CNN + CURVE_FIT IN PARALLELO)
        # -----------------------------------------------------------------
        t_hyb_start = time.perf_counter()
        hybrid_results = parallel(
            delayed(fit_single_wrapper)(i, noisy_sig[i], float(nu1_mhz[i]), float(nu2_mhz[i]))
            for i in range(N_spectra_to_run)
        )
        t_hyb_wall = time.perf_counter() - t_hyb_start

        df_hybrid = pd.DataFrame(hybrid_results)
        # Sommiamo il tempo di inferenza CNN per spettro al tempo del curve_fit
        df_hybrid["fit_time_sec"] += t_cnn_per_spec

        # Salvataggio nei dizionari in RAM
        predictions_hybrid["c"][(c_val, n_ph)] = df_hybrid["c_fit_norm"].to_numpy()
        predictions_hybrid["s"][(c_val, n_ph)] = df_hybrid["s_fit_norm"].to_numpy()

        n_conv = int(df_hybrid["converged"].sum())
        succ_rate = (n_conv / N_spectra_to_run) * 100.0

        save_benchmark_csv(
            file_hybrid,
            df_hybrid,
            meta={
                "Benchmark": "Hybrid (1D-CNN + Levenberg-Marquardt)",
                "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Source_File": os.path.basename(mat_file_path),
                "N_spectra": N_spectra_to_run,
                "N_photons": f"{n_ph:.2e}",
                "Contrast": c_val,
                "Converged_Spectra": f"{n_conv}/{N_spectra_to_run} ({succ_rate:.2f}%)",
                "Mean_MAE_c_norm": f"{df_hybrid['err_c_norm'].mean():.6f}",
                "Mean_MAE_s_norm": f"{df_hybrid['err_s_norm'].mean():.6f}",
                "Total_Wall_Time_s": f"{t_hyb_wall + t_cnn_wall:.2f}",
            },
        )

        print(
            f"Hybrid: {t_hyb_wall:.1f}s | MAE_c(CNN): {df_model['err_c_norm'].mean():.4f} "
            f"-> MAE_c(Hyb): {df_hybrid['err_c_norm'].mean():.4f}"
        )

print(f"\nCompletato in {(time.time() - t_total_start)/60:.1f} minuti!")

Totale combinazioni (Contrast x N_ph): 48
Spettri per combinazione: 10000 (335 punti per spettro)

[1/48] SALTO (già presenti): C=0.8, N_ph=3.0e+05
[2/48] SALTO (già presenti): C=0.8, N_ph=6.0e+05
[3/48] SALTO (già presenti): C=0.8, N_ph=1.2e+06
[4/48] C=0.8  | N_ph=2.4e+06 -> 

CNN: 22.65s | Hybrid: 7.5s | MAE_c(CNN): 0.0001 -> MAE_c(Hyb): 0.0001
[5/48] C=0.8  | N_ph=4.8e+06 -> CNN: 22.75s | Hybrid: 4.8s | MAE_c(CNN): 0.0001 -> MAE_c(Hyb): 0.0001
[6/48] C=0.8  | N_ph=6.0e+06 -> CNN: 23.19s | Hybrid: 5.4s | MAE_c(CNN): 0.0001 -> MAE_c(Hyb): 0.0001
[7/48] C=0.5  | N_ph=3.0e+05 -> CNN: 25.99s | Hybrid: 5.3s | MAE_c(CNN): 0.0004 -> MAE_c(Hyb): 0.0005
[8/48] C=0.5  | N_ph=6.0e+05 -> CNN: 24.52s | Hybrid: 6.1s | MAE_c(CNN): 0.0003 -> MAE_c(Hyb): 0.0003
[9/48] C=0.5  | N_ph=1.2e+06 -> CNN: 30.16s | Hybrid: 5.5s | MAE_c(CNN): 0.0002 -> MAE_c(Hyb): 0.0002
[10/48] C=0.5  | N_ph=2.4e+06 -> CNN: 24.01s | Hybrid: 5.0s | MAE_c(CNN): 0.0002 -> MAE_c(Hyb): 0.0002
[11/48] C=0.5  | N_ph=4.8e+06 -> CNN: 23.71s | Hybrid: 5.1s | MAE_c(CNN): 0.0001 -> MAE_c(Hyb): 0.0001
[12/48] C=0.5  | N_ph=6.0e+06 -> CNN: 23.81s | Hybrid: 5.1s | MAE_c(CNN): 0.0001 -> MAE_c(Hyb): 0.0001
[13/48] C=0.2  | N_ph=3.0e+05 -> CNN: 23.95s | Hybrid: 5.9s | MAE_c(CNN): 0.0012 -> MAE_c(Hyb): 0.0015
[14/48] 